# Januário Leads — Fase 9: captar novas empresas Overture em municípios já cobertos

## Objetivo
A importação anterior atingiu aproximadamente **60 empresas por município**, deixando de fora muitos estabelecimentos. Esta etapa faz uma **nova varredura completa** do Overture em cada cidade, identifica os **IDs empresariais ainda ausentes do banco**, mantém candidatos além dos 100 primeiros resultados brutos e tenta acrescentar **até 40 empresas inéditas** onde já existem 60, chegando a **até 100 por município**.

**Importante:** os 5.570 municípios brasileiros já foram contemplados na malha da primeira etapa; o ganho da Fase 9 será em **empresas novas dentro desses municípios**, e no número de municípios beneficiados. Em municípios com menos de 60 registros, a rotina tenta completar até 100 sempre que houver dados.

### Proteções e lógica
- Destino **exclusivo**: Januário Leads — HOMOLOGAÇÃO (`prlrxjothrslhswpkoza`). Não altera o buscador nem a produção.
- **Não chama Geoapify nem Google Places**. Usa Overture Maps e limites do município da malha oficial do IBGE.
- Ignora rios, lagos e feições geográficas que não são empresas. Puxa os **ID GERS, nome e coordenadas já existentes** da cidade; rejeita IDs repetidos e também candidatos com mesmo nome normalizado a até 15 metros de uma empresa registrada.
- Examina **todo o arquivo de resultados do Overture**, *antes* de selecionar novos candidatos: evita perder os negócios que ficaram de fora da primeira seleção.
- Guarda apenas grupos limitados de novos candidatos para controlar memória e priorizar informações comerciais como telefone, email, site e Instagram, quando disponíveis.
- Máximo de **100 empresas totais por município**; nenhum registro antigo é atualizado. Em colisão de ID, inserção ignora duplicatas.
- Proteção interna: importação **para em 340 MiB** de tamanho do banco; a cota real Supabase Free não foi ampliada.
- **Retomável e nacional:** tenta um município por vez, alternando entre UFs para ampliar a distribuição nacional; evita reconsulta imediata a municípios sem novas empresas usando a tabela privada `overture_phase9_attempts` (homologação).
- `MAX_MUNICIPIOS_SESSAO = 5572` permite percorrer todos os municípios elegíveis na sessão, mas **o Colab interrompe por tempo, erros e espaço antes disso se necessário**. Uma cidade pode conter menos de 100 empresas reais no Overture.
- O Colab só começa a importar depois de executar a última célula e digitar **`AMPLIAR_100`**. Nunca cole credenciais no chat.

## Instruções
1. Execute as **células 1–3**.
2. Verifique que está utilizando a chave **secreta da homologação** e que sua cota Supabase está dentro do limite.
3. Na **célula 4**, digite `AMPLIAR_100` e insira a chave no campo oculto.
4. Envie apenas o relatório `RELATÓRIO FASE 9` (sem chaves). Para retomar, execute novamente a última célula.

**Nenhuma importação é executada automaticamente ao abrir o notebook.**

### Atualização V2 — diagnóstico e recuperação de erros
- Cada falha agora recebe um **código técnico seguro**, gravado no checkpoint `overture_phase9_attempts.error_code`, sem salvar logs brutos, URLs ou chaves.
- Erros transitórios de rede e indisponibilidade podem receber **uma tentativa extra**, com intervalo. Em falhas de autenticação, cota, acesso negado ou armazenamento, o processo **não insiste**.
- Uma sequência de **três falhas** em municípios/UFs diferentes interrompe preventivamente a sessão. Isso evita o padrão da primeira execução, que parou após 12 erros consecutivos.
- O relatório separa causas, tentativas extras e motivo de parada. **Os 12 erros históricos continuam sem código específico**, pois o notebook anterior não registrava os detalhes.
- Se já tinha aberto a versão antiga do Colab, **reabra este notebook e execute novamente as células 1–3** antes de repetir a célula 4.


In [ ]:
# CÉLULA 1 — dependências do importador (somente ao executar)
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências para a Fase 9 disponíveis.")


In [ ]:
# CÉLULA 2 — disponibiliza versão auditável do importador, SEM chamada externa
from pathlib import Path
IMPORTER_PATH=Path("/content/expand_overture_to_100_staging.py")
IMPORTER_SOURCE="#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipalities.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https://<project>.supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/expand_overture_to_100_staging.py --states MG --cities Lavras --database-max-mb 340\n\nPhase 9 top-up only: requires EXACTLY ONE state and ONE city per run, and\naccepts ONLY the staging Supabase project. Read existing IDs first, then insert\nONLY new GERS IDs, without replacing original Overture records. Cap 100 total\nper city, stop at 340 MiB and NEVER run from a browser. No automated trigger.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom urllib import request, error\nfrom urllib.parse import urlparse, urlencode\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ibge.gov.br/organizacao_do_territorio/\"\n    \"malhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef state_boundaries_url(state):\n    \"\"\"IBGE offers one ZIP per UF; do not download the 199 MB national file.\"\"\"\n    if state not in STATE_CODES:\n        raise ValueError(\"Invalid Brazilian state code\")\n    return f\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\ndef ensure_state_boundaries(state, source=None):\n    if source:\n        path = Path(source)\n        if not path.is_file():\n            raise RuntimeError(f\"IBGE boundaries file not found: {path}\")\n        return path\n    path = Path(\"data/ibge/\") / f\"{state}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkdir(parents=True, exist_ok=True)\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloading IBGE municipal boundaries: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n            request.urlretrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise RuntimeError(\"IBGE boundaries download appears empty\")\n            temp.rename(path)\n        finally:\n            temp.unlink(missing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\nSTAGING_URL = \"https://prlrxjothrslhswpkoza.supabase.co\"\nTARGET_PER_MUNICIPALITY = 100\nSTORAGE_STOP_MIB = 340\n# Suppress named geographical features that are not potential companies.\nNON_LEAD_CATEGORIES = frozenset((\n    \"river\", \"lake\", \"stream\", \"waterfall\", \"mountain\", \"cave\",\n    \"island\", \"pond\", \"reservoir\", \"forest\", \"wetland\", \"beach\",\n    \"city\", \"town\", \"village\", \"municipality\", \"county\", \"state\",\n    \"country\", \"natural_feature\", \"neighborhood\", \"locality\",\n))\n\ndef is_lead_candidate(row):\n    return bool(row.get(\"name\") and\n                (row.get(\"category\") or \"\").lower() not in NON_LEAD_CATEGORIES)\n\n\ndef normalize(value):\n    \"\"\"Match the TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in unicodedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    return \" \".join(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in (values or []) if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or link shortener is NOT a verified\n# business website. This preserves truthful \"sem site\" prospecting filters.\nNON_WEBSITE_HOSTS = frozenset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"facebook.com\", \"fb.me\", \"m.me\", \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube.com\", \"youtu.be\",\n    \"tiktok.com\", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai\",\n))\n\n\ndef is_business_website(value):\n    if not isinstance(value, str):\n        return False\n    try:\n        url = urlparse(value.strip())\n        host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n        return (url.scheme in (\"http\", \"https\")\n                and bool(host) and \".\" in host\n                and host not in NON_WEBSITE_HOSTS\n                and not any(host.endswith(\".\" + blocked)\n                            for blocked in NON_WEBSITE_HOSTS))\n    except ValueError:\n        return False\n\n\ndef first_website(values):\n    return first([value for value in (values or []) if is_business_website(value)])\n\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for contactable companies without identified websites.\n\n    Fill unused slots with the best available business contacts. Never promise\n    that a missing website is proof a business actually has no website.\n    \"\"\"\n    ranked = sorted(records, key=lead_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n               and (r.get(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site), (maximum * 2 + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    for row in ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n            chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path, method=\"GET\", payload=None, prefer=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is not None else None\n    # Legacy service_role JWTs use Authorization; the newer sb_secret_ keys\n    # are opaque and must be sent only as apikey, never as Bearer JWT.\n    if not (key.startswith(\"eyJ\") or key.startswith(\"sb_secret_\")):\n        raise RuntimeError(\"Use somente a chave secreta de STAGING (sb_secret_ ou service_role legada).\")\n    headers = {\n        \"apikey\": key,\n        \"Content-Type\": \"application/json\",\n        \"Prefer\": prefer or \"resolution=merge-duplicates,return=minimal\",\n    }\n    if key.startswith(\"eyJ\"):\n        headers[\"Authorization\"] = f\"Bearer {key}\"\n    for attempt in range(5):\n        try:\n            req = request.Request(base + path, method=method, headers=headers, data=data)\n            with request.urlopen(req, timeout=45) as resp:\n                body = resp.read()\n                return json.loads(body) if body else None\n        except error.HTTPError as exc:\n            detail = exc.read(300).decode(\"utf-8\", errors=\"replace\")\n            if exc.code not in (429, 500, 502, 503, 504) or attempt == 4:\n                raise RuntimeError(f\"Supabase API {exc.code}: {detail}\") from exc\n        except (error.URLError, TimeoutError):\n            if attempt == 4:\n                raise\n        time.sleep(min(2 ** attempt, 15))\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef database_bytes(base, key):\n    result = api(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payload={})\n    if not isinstance(result, int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL migration first.\")\n    return result\n\n\ndef existing_city_places(base, key, state, city_key):\n    \"\"\"Read the already imported businesses of one municipality (max 100).\"\"\"\n    query = urlencode({\n        \"select\": \"id,name,name_key,latitude,longitude\",\n        \"state\": \"eq.\" + state, \"city_key\": \"eq.\" + city_key,\n        \"limit\": str(TARGET_PER_MUNICIPALITY + 1),\n    })\n    result = api(base, key, \"/rest/v1/overture_places?\" + query)\n    if not isinstance(result, list):\n        raise RuntimeError(\"Supabase did not return existing city businesses\")\n    if len(result) > TARGET_PER_MUNICIPALITY:\n        raise RuntimeError(\"Municipality exceeds 100 entries. Investigate before continuing.\")\n    return result\n\n\ndef same_business(a, b, meters=15.0):\n    \"\"\"Conservative alias check: EXACT normalized name AND location within 15m.\"\"\"\n    name_a = str(a.get(\"name_key\") or normalize(a.get(\"name\") or \"\"))\n    name_b = str(b.get(\"name_key\") or normalize(b.get(\"name\") or \"\"))\n    if not name_a or name_a != name_b:\n        return False\n    try:\n        la1, lo1 = float(a[\"latitude\"]), float(a[\"longitude\"])\n        la2, lo2 = float(b[\"latitude\"]), float(b[\"longitude\"])\n        if not all(math.isfinite(x) for x in (la1, lo1, la2, lo2)):\n            return False\n        x1, x2, delta_lon = math.radians(la1), math.radians(la2), math.radians(lo2-lo1)\n        delta_lat = x2-x1\n        value = math.sin(delta_lat/2)**2 + math.cos(x1)*math.cos(x2)*math.sin(delta_lon/2)**2\n        distance = 2*6371000*math.asin(min(1, math.sqrt(value)))\n        return distance <= meters\n    except (KeyError, ValueError, TypeError):\n        return False\n\n\ndef matches_any_existing(row, existing):\n    return any(same_business(row, prior) for prior in existing)\n\n\ndef only_new_within_cap(ranked, existing_ids, cap=TARGET_PER_MUNICIPALITY, existing_places=()):\n    \"\"\"Never select existing IDs or near-identical name+location duplicates.\"\"\"\n    room = max(0, cap - len(existing_ids))\n    if room == 0:\n        return []\n    selected, seen = [], set(existing_ids)\n    for row in ranked:\n        identifier = row.get(\"id\")\n        if not identifier or identifier in seen:\n            continue\n        if matches_any_existing(row, existing_places) or matches_any_existing(row, selected):\n            continue\n        seen.add(identifier)\n        selected.append(row)\n        if len(selected) >= room:\n            break\n    return selected\n\n\ndef offer_candidate(groups, row, max_per_group):\n    \"\"\"Bound memory while retaining NEW companies only, across entire input scan.\"\"\"\n    if any(prior.get(\"id\") == row[\"id\"] or same_business(prior, row)\n           for pool in groups.values() for prior in pool):\n        return False\n    pool = groups[\"with_site\" if row.get(\"website\") else \"without_site\"]\n    if len(pool) < max_per_group:\n        pool.append(row)\n        return True\n    worst = min(range(len(pool)), key=lambda idx: lead_quality(pool[idx]))\n    if lead_quality(row) > lead_quality(pool[worst]):\n        pool[worst] = row\n        return True\n    return False\n\n\ndef status(base, key, state, phase, count=0, message=None):\n    api(base, key, \"/rest/v1/overture_import_progress?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase,\n        \"imported_rows\": count, \"last_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n    }])\n\n\ndef municipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = gpd.read_file(\"zip://\" + str(boundaries.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The IBGE municipal boundary file has no CRS.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    columns = set(frame.columns)\n    if not {\"NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError(f\"Unexpected IBGE municipalities columns: {sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(STATE_CODES)].copy().reset_index(drop=True)\n    return frame\n\n\ndef place_record(feature, municipalities, state):\n    from shapely.geometry import Point\n\n    props = feature.get(\"properties\") or {}\n    coords = (feature.get(\"geometry\") or {}).get(\"coordinates\") or []\n    if len(coords) != 2:\n        return None\n    lon, lat = coords\n    if not isinstance(lon, (int, float)) or not isinstance(lat, (int, float)):\n        return None\n    name = (props.get(\"names\") or {}).get(\"primary\")\n    # GeoJSON exporters may place the Overture GERS id in Feature.id rather\n    # than properties.id. Preserve either representation.\n    identifier = props.get(\"id\") or feature.get(\"id\")\n    if not isinstance(name, str) or not name.strip() or not isinstance(identifier, str):\n        return None\n    if props.get(\"operating_status\") == \"permanently_closed\":\n        return None\n\n    # State bounding boxes overlap: map point to the true municipality.\n    point = Point(lon, lat)\n    matches = municipalities.sindex.query(point, predicate=\"intersects\")\n    if len(matches) == 0:\n        return None\n    mun = municipalities.iloc[int(matches[0])]\n    if mun[\"SIGLA_UF\"] != state:\n        return None\n    address = (props.get(\"addresses\") or [{}])[0]\n    if not isinstance(address, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") or {}\n    confidence = props.get(\"confidence\")\n    socials = props.get(\"socials\") or []\n    websites = props.get(\"websites\") or []\n    instagram = first([s for s in (socials + websites)\n                       if isinstance(s, str) and\n                       re.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n        \"id\": identifier[:100],\n        \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],\n        \"category\": str(taxonomy.get(\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        \"city\": str(mun[\"NM_MUN\"])[:120],\n        \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": str(mun[\"SIGLA_UF\"]),\n        \"address\": str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": lat, \"longitude\": lon,\n        \"phone\": (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": (first(props.get(\"emails\")) or \"\")[:200] or None,\n        \"website\": (first_website(websites) or \"\")[:500] or None,\n        \"instagram\": instagram[:500] if instagram else None,\n        \"confidence\": confidence if isinstance(confidence, (int, float)) else None,\n        \"operating_status\": str(props.get(\"operating_status\") or \"\")[:35] or None,\n        \"data_release\": os.getenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n    }\n\n\ndef lead_quality(row):\n    \"\"\"Prioritize genuine, contactable establishments within the per-city quota.\"\"\"\n    confidence = row.get(\"confidence\")\n    weight = float(confidence) * 5 if isinstance(confidence, (int, float)) else 0\n    return (weight\n            + 12 * bool(row.get(\"phone\"))\n            + 11 * bool(row.get(\"email\"))\n            + 7 * bool(row.get(\"website\"))\n            + 3 * bool(row.get(\"instagram\"))\n            + 2 * bool(row.get(\"category\")))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--states\", default=\",\".join(STATE_CODES),\n                        help=\"Comma-separated UF codes; default all 27\")\n    parser.add_argument(\"--cities\", default=\"\",\n                        help=\"Optional comma-separated cities, requires exactly one UF; useful for a small pilot\")\n    parser.add_argument(\"--boundaries\", default=None,\n                        help=\"Optional existing ZIP; defaults to one IBGE ZIP per UF\")\n    parser.add_argument(\"--database-max-mb\", type=int, default=STORAGE_STOP_MIB)\n    parser.add_argument(\"--max-records-per-state\", type=int, default=0,\n                        help=\"0=no state cap; nonzero limits state rows and marks coverage partial\")\n    parser.add_argument(\"--max-records-per-city\", type=int, default=TARGET_PER_MUNICIPALITY,\n                        help=\"Final maximum per municipality: always 100\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = parser.parse_args()\n    if not 30 <= args.database_max_mb <= STORAGE_STOP_MIB:\n        parser.error(\"--database-max-mb must be <= 340 MiB for the top-up pipeline\")\n    if args.max_records_per_state < 0:\n        parser.error(\"--max-records-per-state cannot be negative\")\n    if args.max_records_per_city != TARGET_PER_MUNICIPALITY:\n        parser.error(\"--max-records-per-city must be exactly 100 for this phase\")\n    states = [s.strip().upper() for s in args.states.split(\",\")]\n    if not states or len(set(states)) != len(states) or any(s not in STATE_CODES for s in states):\n        parser.error(\"--states must contain unique Brazilian UF codes\")\n    cities = [normalize(s.strip()) for s in args.cities.split(\",\") if s.strip()]\n    if len(states) != 1 or len(cities) != 1:\n        parser.error(\"Phase 9 processes exactly one municipality from one state per run\")\n\n    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip(\"/\")\n    key = os.getenv(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    if not args.dry_run and (base != STAGING_URL or not key):\n        parser.error(\"Phase 9 accepts ONLY the Januário Leads staging URL and a secret\")\n\n    limit_bytes = args.database_max_mb * 1024 * 1024\n    for state in states:\n        # Individual state archive keeps network and runner-memory use modest.\n        boundaries = ensure_state_boundaries(state, args.boundaries)\n        municipalities = municipalities_index(boundaries)\n        state_munis = municipalities.loc[municipalities.SIGLA_UF == state].copy()\n        if cities:\n            state_munis = state_munis.loc[\n                state_munis.NM_MUN.map(lambda name: normalize(str(name)) in cities)\n            ].copy()\n            found_cities = {normalize(str(name)) for name in state_munis.NM_MUN}\n            if set(cities) != found_cities:\n                raise RuntimeError(f\"Municipality not found in {state}: {sorted(set(cities) - found_cities)}\")\n        if state_munis.empty:\n            raise RuntimeError(f\"IBGE archive does not cover {state}\")\n        west, south, east, north = state_munis.total_bounds\n        # Slightly wider rectangle avoids border rounding gaps.\n        bbox = \",\".join(f\"{v:.6f}\" for v in (west - .005, south - .005, east + .005, north + .005))\n        print(f\"[{state}] Overture bbox {bbox}\", flush=True)\n        if args.dry_run:\n            continue\n        storage_before = database_bytes(base, key)\n        existing_places = existing_city_places(base, key, state, cities[0])\n        existing_ids = {r[\"id\"] for r in existing_places}\n        if len(existing_ids) >= TARGET_PER_MUNICIPALITY:\n            print(f\"ALREADY_COMPLETE: {state}/{cities[0]} has {len(existing_ids)} existing companies.\", flush=True)\n            return 0\n        print(f\"TOP_UP: {state}/{cities[0]} existing={len(existing_ids)}, capacity={TARGET_PER_MUNICIPALITY-len(existing_ids)}.\", flush=True)\n        if storage_before >= limit_bytes:\n            print(f\"STOP: database reached configured {args.database_max_mb} MB safety limit.\")\n            return 2\n        total = 0\n        try:\n            with tempfile.TemporaryDirectory(prefix=\"overture-br-\") as temp:\n                out = Path(temp) / f\"{state}.geojsonseq\"\n                subprocess.run([\"overturemaps\", \"download\", f\"--bbox={bbox}\",\n                                \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(out)],\n                               check=True)\n                # Unlike the first national pass, scan the ENTIRE downloaded\n                # Overture file. Discard the original 60 before ranking newcomers.\n                groups = {\"with_site\": [], \"without_site\": []}\n                counts = {\"scanned\": 0, \"known_id\": 0, \"same_business\": 0,\n                          \"nonbusiness\": 0, \"eligible_new\": 0}\n                with out.open(\"r\", encoding=\"utf-8\") as src:\n                    for line in src:\n                        line = line.lstrip(\"\\x1e\").strip()\n                        if not line:\n                            continue\n                        counts[\"scanned\"] += 1\n                        row = place_record(json.loads(line), state_munis, state)\n                        if row is None or row[\"city_key\"] != cities[0]:\n                            continue\n                        if not is_lead_candidate(row):\n                            counts[\"nonbusiness\"] += 1\n                            continue\n                        if row[\"id\"] in existing_ids:\n                            counts[\"known_id\"] += 1\n                            continue\n                        if matches_any_existing(row, existing_places):\n                            counts[\"same_business\"] += 1\n                            continue\n                        counts[\"eligible_new\"] += 1\n                        offer_candidate(groups, row, args.max_records_per_city)\n\n                capacity = TARGET_PER_MUNICIPALITY - len(existing_ids)\n                # Both pools include only newly discovered IDs and exclude\n                # obvious same-name/same-coordinate duplicates.\n                ranked = select_diverse_pool(\n                    groups[\"with_site\"] + groups[\"without_site\"], capacity\n                )\n                selected = only_new_within_cap(ranked, existing_ids,\n                                               existing_places=existing_places)\n                print(f\"SCAN: {counts['scanned']} Overture rows; \"\n                      f\"{counts['known_id']} existing IDs excluded; \"\n                      f\"{counts['nonbusiness']} non-commercial places ignored; \"\n                      f\"{counts['same_business']} same-business aliases excluded; \"\n                      f\"{counts['eligible_new']} candidate new rows; \"\n                      f\"{len(selected)} selected unique companies.\", flush=True)\n                if not selected:\n                    print(f\"NO_NEW_PLACES: {state}/{cities[0]} has no additional eligible Overture companies.\", flush=True)\n                    return 0\n                # A municipality-only pilot never marks the whole state as imported.\n                partial = bool(cities)\n                if args.max_records_per_state and len(selected) > args.max_records_per_state:\n                    selected = selected[:args.max_records_per_state]\n                    partial = True\n                for start in range(0, len(selected), BATCH_SIZE):\n                    if database_bytes(base, key) >= limit_bytes:\n                        partial = True\n                        break\n                    # Second read protects against a rerun or another writer\n                    # that added new records after this city's initial check.\n                    fresh = existing_city_places(base, key, state, cities[0])\n                    fresh_ids = {r[\"id\"] for r in fresh}\n                    batch = only_new_within_cap(\n                        selected[start:start + BATCH_SIZE], fresh_ids,\n                        existing_places=fresh)\n                    if not batch:\n                        break\n                    api(base, key, \"/rest/v1/overture_places?on_conflict=id\", \"POST\", batch,\n                        prefer=\"resolution=ignore-duplicates,return=minimal\")\n                    total += len(batch)\n                storage_after = database_bytes(base, key)\n                delta = storage_after - storage_before\n                bytes_per_imported_row = delta // total if total else 0\n                print(f\"[{state}] {total} imported; \"\n                      f\"database before={storage_before}, after={storage_after}, \"\n                      f\"delta={delta}, average={bytes_per_imported_row} B/row; \"\n                      f\"{'PARTIAL' if partial else 'COMPLETE'}\", flush=True)\n                if storage_after >= limit_bytes:\n                    print(\"STOP: database safety threshold; remaining states NOT imported.\", flush=True)\n                    return 2\n        except Exception:\n            raise\n    print(\"Ingestion pass ended. Check overture_import_progress for coverage.\", flush=True)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
IMPORTER_PATH.write_text(IMPORTER_SOURCE,encoding="utf-8")
compile(IMPORTER_SOURCE,str(IMPORTER_PATH),"exec")
print("Importador Overture fase 9 salvo localmente e sintaxe validada. Nenhuma importação.")


In [ ]:
# CÉLULA 3 — configurações e testes locais; ZERO consultas externas
import datetime as dt
import getpass
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from urllib.parse import urlencode
from urllib.request import Request,urlopen
from urllib.error import HTTPError

STAGING_URL="https://prlrxjothrslhswpkoza.supabase.co"
TARGET_PER_CITY=100
MAX_MUNICIPIOS_SESSAO=5572  # percorre todos os elegíveis; para por tempo e espaço
STORAGE_STOP_MIB=340
TIME_LIMIT_MINUTES=540
MAX_ERRORS=12
MAX_CONSECUTIVE_ERRORS=3
MAX_DIFFERENT_UFS_WITH_FAILURES=3
PAGE_SIZE=1000
PAUSE_SECONDS=1
TRANSIENT_RETRY_SECONDS=8
FAILURE_BACKOFF_SECONDS=(4,12)
MIB=1024*1024
assert TARGET_PER_CITY==100 and STORAGE_STOP_MIB==340
assert 1<=MAX_MUNICIPIOS_SESSAO<=5572 and 30<=TIME_LIMIT_MINUTES<=600
assert IMPORTER_PATH.is_file() and IMPORTER_SOURCE

def staged_rest(secret,path,method="GET",body=None):
    if not path.startswith("/rest/v1/") or path.startswith("/rest/v1//"):
        raise ValueError("Apenas rotas PostgREST são permitidas")
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    data=json.dumps(body).encode("utf-8") if body is not None else None
    req=Request(STAGING_URL+path,method=method,data=data,headers=headers)
    with urlopen(req,timeout=45) as response:
        raw=response.read(2000000)
        return json.loads(raw.decode("utf-8")) if raw else None

def storage_mib(secret):
    raw=staged_rest(secret,"/rest/v1/rpc/overture_storage_bytes","POST",{})
    if not isinstance(raw,int):raise RuntimeError("Falha no controle de armazenamento")
    return raw/MIB

def municipality_rows(secret):
    out=[]
    offset=0
    while True:
        qs=urlencode({"select":"state,city,city_key,imported_rows",
                      "order":"state.asc,city_key.asc",
                      "limit":str(PAGE_SIZE),"offset":str(offset)})
        page=staged_rest(secret,"/rest/v1/overture_import_city_progress?"+qs)
        if not isinstance(page,list):raise RuntimeError("Não foi possível listar municípios")
        out.extend(page)
        if len(page)<PAGE_SIZE:break
        offset+=len(page)
    return out

def pending_cities(secret):
    """Read ONLY the Phase 9 queue, never the Phase 1–7 retry schedule."""
    now=dt.datetime.now(dt.timezone.utc).isoformat()
    result=set()
    offset=0
    while True:
        qs=urlencode({"select":"state,city_key","retry_after":"gt."+now,
                      "order":"state.asc,city_key.asc","limit":str(PAGE_SIZE),
                      "offset":str(offset)})
        page=staged_rest(secret,"/rest/v1/overture_phase9_attempts?"+qs)
        if not isinstance(page,list):raise RuntimeError("Falha ao verificar retomadas da Fase 9")
        result.update((r["state"],r["city_key"]) for r in page)
        if len(page)<PAGE_SIZE:break
        offset+=len(page)
    return result

def sorted_targets(rows,pending=(),tried=()):
    """Balance UFs so a few very large states do not monopolize the session."""
    blocked=set(pending)|set(tried)
    buckets={}
    for r in rows:
        if not isinstance(r,dict):continue
        n=r.get("imported_rows")
        uf=r.get("state")
        if (not isinstance(n,int) or n<=0 or n>=TARGET_PER_CITY
                or not uf or not r.get("city") or not r.get("city_key")
                or (uf,r["city_key"]) in blocked):continue
        buckets.setdefault(uf,[]).append(r)
    for uf in buckets:
        buckets[uf].sort(key=lambda r:(-r["imported_rows"],r["city_key"]))
    result=[]
    # Round robin across states, favoring towns at the old 60-company cap.
    states=sorted(buckets)
    for i in range(max((len(v) for v in buckets.values()),default=0)):
        for uf in states:
            if i<len(buckets[uf]):result.append(buckets[uf][i])
    return result

def record_attempt(secret,state,city,city_key,reason,before,after,error_code=None):
    """Private checkpoint, never mutates any Overture company or national progress."""
    if reason not in ("gained","no_more","error","partial"):
        raise ValueError("Invalid Phase 9 checkpoint status")
    from datetime import timedelta
    delay={"gained":7,"no_more":14,"error":0.25,"partial":0.25}[reason]
    due=(dt.datetime.now(dt.timezone.utc)+timedelta(days=delay)).isoformat()
    if error_code is not None and not re.fullmatch(r"[a-z0-9_]{1,80}",error_code):
        raise ValueError("Código de erro fora do padrão seguro")
    payload={"state":state,"city":city,"city_key":city_key,
             "result":reason,"before_count":before,"after_count":after,
             "added_count":max(0,after-before),"retry_after":due,
             "error_code":error_code}
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json",
             "Prefer":"resolution=merge-duplicates,return=minimal"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    req=Request(STAGING_URL+"/rest/v1/overture_phase9_attempts?on_conflict=state,city_key",
                method="POST",data=json.dumps(payload).encode("utf-8"),headers=headers)
    with urlopen(req,timeout=35) as resp:resp.read(20000)

def actual_city_count(secret,state,city_key):
    query=urlencode({"select":"id","state":"eq."+state,"city_key":"eq."+city_key})
    headers={"apikey":secret,"Accept":"application/json","Prefer":"count=exact"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    req=Request(STAGING_URL+"/rest/v1/overture_places?"+query,headers=headers,method="HEAD")
    with urlopen(req,timeout=45) as resp:
        number=(resp.headers.get("Content-Range") or "").rsplit("/",1)[-1]
        if not number.isdigit():raise RuntimeError("Não foi possível confirmar a contagem")
        return int(number)

# Classifica SOMENTE a categoria técnica. Nunca imprime stdout/stderr bruto,
# URLs, endpoints com apiKey, tokens, corpo de resposta, nem traceback completo.
def classify_import_error(output,returncode=None,timed_out=False):
    if timed_out:
        return "timeout_subprocesso"
    content=str(output or "").lower()[-12000:]
    if "database safety threshold" in content or "database reached configured" in content:
        return "limite_banco"
    if "supabase api 401" in content or "supabase api 403" in content or "jwt expired" in content:
        return "supabase_auth"
    if "supabase api 429" in content:
        return "supabase_taxa"
    if any(s in content for s in ("http error 429","http 429","too many requests","throttl","rate limit","http 429")):
        return "fonte_taxa"
    if any(s in content for s in ("http error 403","http error 401","accessdenied","access denied","forbidden")):
        return "fonte_acesso"
    if any(s in content for s in ("http error 503","http error 502","http error 504","service unavailable","bad gateway","gateway timeout")):
        return "fonte_indisponivel"
    if "supabase api" in content and any(s in content for s in (" 500"," 502"," 503"," 504")):
        return "supabase_indisponivel"
    if any(s in content for s in ("no module named","modulenotfounderror","importerror","command not found","filenotfounderror")):
        return "dependencia_ausente"
    if any(s in content for s in ("out of memory","memoryerror","cannot allocate memory","killed")):
        return "sem_memoria"
    if any(s in content for s in ("certificate verify failed","sslerror","tls error")):
        return "tls"
    if any(s in content for s in ("timed out","timeouterror","readtimeout","connecttimeout")):
        return "rede_timeout"
    if any(s in content for s in ("connection reset","connection refused","network is unreachable","name resolution","dns","urlerror")):
        return "rede_conexao"
    if any(s in content for s in ("municipality not found","unexpected ibge","ibge archive does not cover","boundaries download")):
        return "malha_ibge"
    if any(s in content for s in ("jsondecodeerror","invalid json","valueerror")):
        return "dados_invalidos"
    if "calledprocesserror" in content or "overturemaps download" in content:
        return "download_overture"
    if "supabase api" in content:
        return "supabase_http"
    if returncode is not None and returncode<0:
        return "processo_sinal"
    return "falha_desconhecida"

def stop_on_global_error(code):
    return code in ("limite_banco","supabase_auth","fonte_acesso","supabase_taxa",
                    "fonte_taxa","dependencia_ausente","sem_memoria","tls")

def transient_error(code):
    return code in ("rede_timeout","rede_conexao","fonte_indisponivel",
                    "supabase_indisponivel","timeout_subprocesso")

def failure_backoff(streak):
    return FAILURE_BACKOFF_SECONDS[min(max(streak-1,0),len(FAILURE_BACKOFF_SECONDS)-1)]

# Testes que não dependem de chave nem de acesso à internet.
sample_rows=[
    {"state":"MG","city":"Lavras","city_key":"lavras","imported_rows":60},
    {"state":"SP","city":"Santos","city_key":"santos","imported_rows":100},
    {"state":"RJ","city":"Niterói","city_key":"niteroi","imported_rows":22},
]
assert [r["city_key"] for r in sorted_targets(sample_rows)]==["lavras","niteroi"]
assert [r["city_key"] for r in sorted_targets(sample_rows+[{"state":"MG","city":"Alfenas","city_key":"alfenas","imported_rows":59}])]==["lavras","niteroi","alfenas"]
assert [r["city_key"] for r in sorted_targets(sample_rows,pending={("MG","lavras")})]==["niteroi"]
assert TARGET_PER_CITY-60==40 and TARGET_PER_CITY-100==0
assert classify_import_error("Supabase API 401: secret_here")== "supabase_auth"
assert classify_import_error("HTTP Error 429: throttled")=="fonte_taxa"
assert classify_import_error("connection reset by peer")=="rede_conexao"
assert classify_import_error("private url",returncode=1)=="falha_desconhecida"
assert classify_import_error("",timed_out=True)=="timeout_subprocesso"
assert stop_on_global_error("fonte_taxa") and not stop_on_global_error("rede_timeout")
assert transient_error("rede_timeout") and failure_backoff(3)==12
print("TESTES OFFLINE OK: meta 100, retomada, diagnóstico e proteção de erros.")


In [ ]:
# CÉLULA 4 — EXPANSÃO MANUAL SOMENTE NA HOMOLOGAÇÃO
def ampliar_100():
    print("FASE 9 — OVERTURE: complementar municípios existentes de 60 para até 100 empresas.")
    print("Teto de armazenamento:",STORAGE_STOP_MIB,"MiB | municípios por sessão:",MAX_MUNICIPIOS_SESSAO)
    confirm=input("Para INICIAR a importação, digite exatamente AMPLIAR_100: ").strip()
    if confirm!="AMPLIAR_100":
        print("CANCELADO: nenhuma importação executada.")
        return
    secret=getpass.getpass("Chave SECRETA da HOMOLOGAÇÃO (campo oculto): ").strip()
    if not (secret.startswith("sb_secret_") or secret.startswith("eyJ")):
        print("Credencial ausente ou inválida; nenhuma importação executada.")
        return
    imported=0; completed=0; no_more=0; errors=0; skipped=0; attempted=0; improved=0
    consecutive_errors=0
    recent_failed_states=[]
    failure_categories={}
    pause_reason=None
    retries=0
    try:
        used=storage_mib(secret)
        print("Banco de homologação:",round(used,2),"MiB")
        if used>=STORAGE_STOP_MIB:
            print("PAUSA: banco no limite preventivo.")
            return
        rows=municipality_rows(secret)
        pending=pending_cities(secret)
        targets=sorted_targets(rows,pending)
        print("Municípios já existentes:",len(rows),"| abaixo de 100:",sum(r.get("imported_rows",0)<100 for r in rows),
              "| elegíveis fora da fila de espera:",len(targets))
        deadline=time.monotonic()+TIME_LIMIT_MINUTES*60
        for row in targets:
            if attempted>=MAX_MUNICIPIOS_SESSAO or errors>=MAX_ERRORS:
                if errors>=MAX_ERRORS:pause_reason="teto_de_erros_sessao"
                break
            if time.monotonic()>deadline-90:
                print("PAUSA: limite de tempo desta sessão.")
                break
            if storage_mib(secret)>=STORAGE_STOP_MIB:
                print("PAUSA: proteção de armazenamento (340 MiB).")
                break
            state,city,key=row["state"],row["city"],row["city_key"]
            before=actual_city_count(secret,state,key)
            if before>=TARGET_PER_CITY:
                skipped+=1
                continue
            attempted+=1
            print("INICIANDO:",city+"/"+state,"| existentes:",before,"| faltantes até 100:",TARGET_PER_CITY-before,flush=True)
            command=[sys.executable,str(IMPORTER_PATH),
                     "--states",state,"--cities",city,
                     "--max-records-per-city","100",
                     "--database-max-mb",str(STORAGE_STOP_MIB)]
            env=os.environ.copy()
            env["SUPABASE_URL"]=STAGING_URL
            env["SUPABASE_SERVICE_ROLE_KEY"]=secret
            run=None
            error_code=None
            timed_out=False
            output=""
            left=int(deadline-time.monotonic()-60)
            if left<90:
                pause_reason="tempo_de_sessao"
                print("PAUSA: tempo de sessão insuficiente para novo download.")
                break
            # A segunda tentativa só ocorre em falhas transitórias, com atraso.
            # Nunca repete 401/403, 429, indisponibilidade de armazenamento ou
            # problemas de configuração; evita piorar bloqueios do provedor.
            for trial in range(2):
                try:
                    run=subprocess.run(command,env=env,text=True,capture_output=True,
                                       timeout=min(1200,max(90,int(deadline-time.monotonic()-60))))
                    output=(run.stdout or "")+"\\n"+(run.stderr or "")
                    if run.returncode==0:
                        error_code=None
                        break
                    error_code=classify_import_error(output,run.returncode)
                except subprocess.TimeoutExpired:
                    timed_out=True
                    run=None
                    error_code=classify_import_error("",timed_out=True)
                if trial==0 and transient_error(error_code) and time.monotonic()+TRANSIENT_RETRY_SECONDS+120<deadline:
                    retries+=1
                    print("RECUPERAÇÃO:",city+"/"+state,"| categoria:",error_code,
                          "| 1 nova tentativa controlada.")
                    time.sleep(TRANSIENT_RETRY_SECONDS)
                    continue
                break
            # A contagem do banco é a única prova de inserção, não a saída do importador.
            after=actual_city_count(secret,state,key)
            gained=max(0,after-before)
            imported+=gained
            if gained:improved+=1
            if after>100:
                pause_reason="contagem_acima_de_100"
                print("ALERTA: contagem acima de 100. Parada obrigatória:",city+"/"+state)
                break
            if after>=100:completed+=1
            if gained:
                print("SALVO:",city+"/"+state,"| +",gained,"empresas inéditas | total:",after,"/100")
            if error_code is not None:
                errors+=1
                consecutive_errors+=1
                recent_failed_states.append(state)
                recent_failed_states=recent_failed_states[-5:]
                failure_categories[error_code]=failure_categories.get(error_code,0)+1
                print("FALHA DIAGNOSTICADA:",city+"/"+state,
                      "| categoria:",error_code,
                      "| saída do processo:",run.returncode if run is not None else "timeout",
                      "| sequência:",consecutive_errors)
            else:
                consecutive_errors=0
                recent_failed_states=[]
                if not gained:
                    no_more+=1
                    print("SEM NOVOS DADOS:",city+"/"+state,
                          "| Overture não encontrou empresa adicional")

            result=("partial" if error_code and gained else "error" if error_code
                    else "gained" if gained else "no_more")
            try:
                record_attempt(secret,state,city,key,result,before,after,error_code)
            except Exception as exc:
                errors+=1
                pause_reason="checkpoint_indisponivel"
                print("PAUSA: checkpoint não pôde ser salvo. Tipo:",type(exc).__name__)
                break

            if error_code is not None:
                # Em três falhas seguidas ou 3 UFs com falhas entre as últimas
                # cinco tentativas, suspende a sessão para diagnóstico.
                if stop_on_global_error(error_code):
                    pause_reason="erro_global_"+error_code
                elif consecutive_errors>=MAX_CONSECUTIVE_ERRORS:
                    pause_reason="falhas_consecutivas"
                elif len(set(recent_failed_states))>=MAX_DIFFERENT_UFS_WITH_FAILURES:
                    pause_reason="falhas_em_ufs_distintas"
                if pause_reason:
                    print("PAUSA PROTETIVA:",pause_reason,
                          "| procure corrigir a causa antes de retomar.")
                    break
                delay=failure_backoff(consecutive_errors)
                if time.monotonic()+delay<deadline:
                    print("ESPERA CONTROLADA APÓS ERRO:",delay,"segundos")
                    time.sleep(delay)
            elif run is not None and ("STOP: database" in (run.stdout or "")
                                      or "database safety threshold" in (run.stdout or "")):
                pause_reason="limite_banco"
                print("PAUSA: limite preventivo reportado pelo importador.")
                break
            time.sleep(PAUSE_SECONDS)
        print("\n"+"="*58)
        print("RELATÓRIO FASE 9 — COMPLEMENTO OVERTURE ATÉ 100")
        print("Municípios tentados nesta sessão:",attempted)
        print("Municípios que ganharam pelo menos 1 empresa:",improved)
        print("Municípios que chegaram a 100 nesta sessão:",completed)
        print("Empresas novas confirmadas por contagem:",imported)
        print("Municípios sem novas empresas na fonte:",no_more)
        print("Municípios temporariamente adiados por checkpoint:",len(pending))
        print("Municípios já completos ignorados:",skipped)
        print("Erros nesta sessão:",errors)
        print("Tentativas extras por falhas transitórias:",retries)
        print("Motivos dos erros:",dict(sorted(failure_categories.items())))
        print("Motivo da parada:",pause_reason or "limite_de_lote_ou_fila_esgotada")
        print("Banco ao final:",round(storage_mib(secret),2),"MiB")
        print("Nenhum dado anterior foi sobrescrito. Não houve requisições Geoapify/Google.")
        print("Se houver falhas consecutivas, revise o motivo antes de reexecutar.")
        print("Para continuar após essa análise, execute novamente APENAS esta célula.")
        print("="*58)
    except HTTPError as exc:
        print("Falha HTTP na HOMOLOGAÇÃO:",exc.code,"— nenhuma chave exibida.")
    finally:
        secret=""
        print("Credenciais descartadas do notebook. Desconecte o runtime ao terminar.")

ampliar_100()
